In [1]:
import os
import wandb

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

import numpy as np
import pandas as pd
import random
from tqdm import *

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets
import torchvision.transforms as transforms

try:
    from torchinfo import summary
except:
    print("[INFO] Couldn't find torchinfo... installing it.")
    !pip install -q torchinfo
    from torchinfo import summary

from PIL import Image

import matplotlib.pyplot as plt
%matplotlib inline

### Зафиксируем seed для воспроизводимости

In [2]:
def seed_everything(seed):
    random.seed(seed) # фиксируем генератор случайных чисел
    os.environ['PYTHONHASHSEED'] = str(seed) # фиксируем заполнения хешей
    np.random.seed(seed) # фиксируем генератор случайных чисел numpy
    torch.manual_seed(seed) # фиксируем генератор случайных чисел pytorch
    torch.cuda.manual_seed(seed) # фиксируем генератор случайных чисел для GPU
    torch.backends.cudnn.deterministic = True # выбираем только детерминированные алгоритмы (для сверток)
    torch.backends.cudnn.benchmark = False # фиксируем алгоритм вычисления сверток

### Задаем параметры (конфиг)

In [3]:
all_brands = ('Audi', 'BMW', 'Chevrolet', 'Hyundai', 'Kia', 'Lexus', 'Mercedes-Benz', 'Toyota')
all_body_types = ('sedan', 'SUV')

class CFG:
  api = ""
  project = "kolesa-cars"
  entity = "armntvs-d3v-student"
  num_epochs = 10
  train_batch_size = 32
  test_batch_size = 512
  num_workers = 0
  lr = 3e-4
  seed = 42
  classes = all_body_types
  wandb = True

In [4]:
# вход в W&B (ключ спросит один раз, берём с https://wandb.ai/authorize)
wandb.login()


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/ksa/.netrc.
wandb: Currently logged in as: armntvs-d3v (armntvs-d3v-student) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
seed_everything(CFG.seed)

In [6]:
# Переведем наш класс с параметрами в словарь

def class2dict(f):
  return dict((name, getattr(f, name)) for name in dir(f) if not name.startswith('__'))

### Импортируем наш датасет с картинками

In [ ]:
try:
  from google.colab import drive
  drive.mount('/content/drive')
  !cp -r /content/drive/MyDrive/gp5_project/data /content/data

  base_dir = '/content/data'
except:
  base_dir = '../data'

from pathlib import Path

base_dir = Path(base_dir)

In [8]:
df_bt = pd.read_csv(base_dir / 'prepared_df_images.csv')
df_bt

,kolesa_id,body_type,img_filename,target
0,220599328,Седан,220599328.webp,0
1,222486528,Седан,222486528.webp,0
2,218695550,Внедорожник,218695550.webp,1
3,214781208,Седан,214781208.webp,0
4,222346737,Седан,222346737.webp,0
...,...,...,...,...
4428,222478046,Седан,222478046.webp,0
4429,222704181,Седан,222704181.webp,0
4430,220000872,Седан,220000872.webp,0
4431,222542135,Седан,222542135.webp,0


In [9]:
df_brand = pd.read_csv(base_dir / 'brand_df_images.csv')
df_brand

,kolesa_id,brand,body_type,img_filename,target
0,220599328,Chevrolet,Седан,220599328.webp,2
1,222486528,Chevrolet,Седан,222486528.webp,2
2,218695550,Chevrolet,Внедорожник,218695550.webp,2
3,214781208,Chevrolet,Седан,214781208.webp,2
4,222346737,Chevrolet,Седан,222346737.webp,2
...,...,...,...,...,...
4428,222478046,Hyundai,Седан,222478046.webp,3
4429,222704181,Hyundai,Седан,222704181.webp,3
4430,220000872,Hyundai,Седан,220000872.webp,3
4431,222542135,Hyundai,Седан,222542135.webp,3


In [10]:
train_df_bt, test_all_df_bt = train_test_split(df_bt, test_size=0.3, stratify=df_bt['target'], random_state=CFG.seed)
val_df_bt, test_df_bt = train_test_split(test_all_df_bt, test_size=0.5, stratify=test_all_df_bt['target'], random_state=CFG.seed)

train_df_brand, test_all_df_brand = train_test_split(df_brand, test_size=0.3, stratify=df_brand['target'], random_state=CFG.seed)
val_df_brand, test_df_brand = train_test_split(test_all_df_brand, test_size=0.5, stratify=test_all_df_brand['target'], random_state=CFG.seed)

In [11]:
# https://stackoverflow.com/questions/65231299/load-csv-and-image-dataset-in-pytorch

class KolesaDataset(torch.utils.data.Dataset):
    def __init__(self, df, images_folder = base_dir / 'images', transform = None):
        # т.к. после train_test_split индексы сохряняются из исходного датасета, то мы их сбрасываем и удаляем колонку со старыми индексами для корректной работы __getitem___
        self.df = df.reset_index().drop(columns=['index'])
        self.images_folder = images_folder
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        filename = self.df.loc[index, 'img_filename']
        label = self.df.loc[index, 'target']
        image = Image.open(os.path.join(self.images_folder, filename)).convert('RGB')

        if self.transform is not None:
            image = self.transform(image)

        return image, label



### Функции обучения и инференса

In [12]:
# функция обучения
def train(model, device, train_loader, optimizer, criterion, epoch, WANDB):
    model.train()
    train_loss = 0
    correct = 0

    n_ex = len(train_loader)

    for batch_idx, (data, target) in tqdm(enumerate(train_loader), total=n_ex):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad() # обнуляем градиенты
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)
        correct += pred.eq(target.view_as(pred)).sum().item()
        train_loss = criterion(output, target) # считаем лосс
        train_loss.backward() # обратный проход
        optimizer.step() # делаем шаг оптимизатором

    tqdm.write('\nTrain set: Average loss: {:.4f}, Accuracy: {:.2f}%'.format(
        train_loss, 100. * correct / len(train_loader.dataset)))

    # логируем функцию потерь и точность
    if WANDB:
        wandb.log({'train_loss': train_loss,
                   'train_accuracy': correct / len(train_loader.dataset)})

In [13]:
# функция инференса
def test(model, device, test_loader, criterion, WANDB):
    model.eval()
    test_loss = 0
    correct = 0

    preds = []
    targets = []
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss = criterion(output, target)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

            # https://stackoverflow.com/questions/53467215/convert-pytorch-cuda-tensor-to-numpy-array
            preds.extend(pred.cpu().numpy())
            targets.extend(target.cpu().numpy())

    # https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
    report = classification_report(targets, preds, target_names=CFG.classes)

    accuracy = correct / len(test_loader.dataset)
    macro_f1 = f1_score(targets, preds, average='macro')
    weighted_f1 = f1_score(targets, preds, average='weighted')

    tqdm.write('Test set: Accuracy: {:.2f}%'.format(100. * accuracy))

    tqdm.write('\nClassification report:\n' + report)

    if WANDB:
        wandb.log({'test_loss': accuracy, 'test_macro_f1': macro_f1, 'test_weighted_f1': weighted_f1})

    return accuracy, macro_f1, weighted_f1, report

# функция инференса
def val(model, device, val_loader, criterion, WANDB):
    model.eval()
    val_loss = 0
    correct = 0

    preds = []
    targets = []
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            val_loss = criterion(output, target)
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

            preds.extend(pred.cpu().numpy())
            targets.extend(target.cpu().numpy())

    report = classification_report(targets, preds, target_names=CFG.classes)

    tqdm.write('Val set: Average loss: {:.4f}, Accuracy: {:.2f}%'.format(
        val_loss, 100. * correct / len(val_loader.dataset)))

    tqdm.write('\nClassification report:\n' + report)

    if WANDB:
        wandb.log({'val_loss': val_loss,
                   'val_accuracy': correct / len(val_loader.dataset)})

### Основная функция для прогона моделей

In [14]:
def main_kolesa(model, train_df, val_df):

    if CFG.wandb:
        wandb.init(project=CFG.project, entity=CFG.entity, reinit=True, config=class2dict(CFG))
        # логируем архитектуру модели
        # https://docs.wandb.ai/guides/track/config
        wandb.config.update({'model': str(model)})

    seed_everything(CFG.seed)
    # https://stackoverflow.com/questions/63423463/using-pytorch-cuda-on-macbook-pro
    # т.к. на macbook на их процессорах apple silicon нет cuda (только для карт nvidia), то используем альтеранативу, но если есть cuda - то используем ее
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')

    kwargs = {'num_workers': CFG.num_workers, 'pin_memory': torch.cuda.is_available()}

    # наши картинки в исходном размере 750*470 + если домножить на число каналов (3), то получится около миллиарда параметра на вход.
    # Это довольно много так что пропорционально уменьшим размерность. У нас 750 к 470 это, можно сказать, 1.6 к 1. Уменьшим 750 до 160 => 470 -> 96.
    # будем использовать различные методы аугментации https://docs.pytorch.org/vision/0.13/transforms.html
    train_transform = transforms.Compose([
        transforms.Resize((96, 160)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225)
        )
        ])

    test_transform = transforms.Compose([
        transforms.Resize((96, 160)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225)
        )
        ])

    train_ds = KolesaDataset(train_df, transform=train_transform)
    val_ds = KolesaDataset(val_df, transform=test_transform)
    # test_ds = KolesaDataset(test_df, transform=test_transform)

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=CFG.train_batch_size, **kwargs)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=CFG.test_batch_size, **kwargs)
    # test_loader = torch.utils.data.DataLoader(test_ds, batch_size=CFG.test_batch_size, **kwargs)

    model = model.to(device)

    if CFG.wandb:
        wandb.watch(model, log='all')

    optimizer = optim.Adam(model.parameters(),
                          lr=CFG.lr)

    criterion = nn.CrossEntropyLoss()

    for epoch in range(1, CFG.num_epochs + 1):
        print('\nEpoch:', epoch)
        train(model, device, train_loader, optimizer, criterion, epoch, CFG.wandb)
        val(model, device, val_loader, criterion, CFG.wandb)

    print('Training is ended!')

### Сохранение моделей

Чтобы у каждой модели сохранялся артефакт, добавим маленькую вспомогательную функцию. `main_kolesa` логирует loss/accuracy по эпохам в W&B сам, а эта функция дописывает в тот же запуск файл с весами.

In [15]:
def save_model_artifact(model, name):
    # сохранение модели как артефакт
    # https://docs.wandb.ai/guides/artifacts
    path = f'../data/models/{name}.pt' #путь к файлу в зависимости от имени модели (передаем в начале), .pt -расширение для объектов pytorch
    torch.save(model.state_dict(), path) #сохраняем параметры модели
    if CFG.wandb:
        wandb.run.name = name   # называем запуск как модель
        artifact = wandb.Artifact(name, type='model') #создаем артефакт-контейннер
        artifact.add_file(path) #добавляем в него файл с параметрами модели (брали выше)
        wandb.log_artifact(artifact) #загружаем артефакт и привящываем к текущему щапуску
        wandb.finish()  # закрываем текущий запуск
    print(f'[OK] модель сохранена: {path}')

In [ ]:
# данные эксперимента в W&B
# https://docs.wandb.ai/guides/artifacts
if CFG.wandb:
    wandb.init(project=CFG.project, entity=CFG.entity, reinit=True, name='dataset')
    art = wandb.Artifact('kolesa-data', type='dataset')
    art.add_file('../data/prepared_df_images.csv')
    art.add_file('../data/brand_df_images.csv')
    art.add_dir('../data/images', name='images')
    wandb.log_artifact(art)
    wandb.finish()


## Fully Connected в качестве baseline

Возьмем для примера определение весов нормальным распределением

In [16]:
# определяем полносвязанную сеть
class FC_Net(nn.Module):
    def __init__(self, task='body_type'):
        super(FC_Net, self).__init__()

        self.task = task

        self.fc1 = nn.Linear(160*96*3, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, 128)
        if self.task == 'body_type':
            self.fc4 = nn.Linear(128, 2)
        elif self.task == 'brand':
            self.fc4 = nn.Linear(128, 8)

        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = x.view(-1, 160*96*3)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x

#### Задача на тип кузова

In [17]:
model_baseline_bt = FC_Net()

In [18]:
summary(model=model_baseline_bt,
        input_size=(32, 3, 96, 160), # входной батч
        col_names=["input_size", "output_size", "num_params", "trainable"], # что хотим посмотреть
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
FC_Net                                   [32, 3, 96, 160]     [32, 2]              --                   True
├─Linear: 1-1                            [32, 46080]          [32, 1024]           47,186,944           True
├─Linear: 1-2                            [32, 1024]           [32, 512]            524,800              True
├─Linear: 1-3                            [32, 512]            [32, 128]            65,664               True
├─Linear: 1-4                            [32, 128]            [32, 2]              258                  True
Total params: 47,777,666
Trainable params: 47,777,666
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 1.53
Input size (MB): 5.90
Forward/backward pass size (MB): 0.43
Params size (MB): 191.11
Estimated Total Size (MB): 197.44

In [19]:
main_kolesa(model_baseline_bt, train_df = train_df_bt, val_df = val_df_bt)
save_model_artifact(model_baseline_bt, 'BaseLine_FC_body_type')

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.



Epoch: 1


100%|██████████| 97/97 [00:18<00:00,  5.34it/s]



Train set: Average loss: 200857.7188, Accuracy: 54.04%
Val set: Average loss: 133297.0781, Accuracy: 54.74%

Classification report:
              precision    recall  f1-score   support

       sedan       0.66      0.58      0.62       418
         SUV       0.41      0.50      0.45       247

    accuracy                           0.55       665
   macro avg       0.54      0.54      0.53       665
weighted avg       0.57      0.55      0.55       665


Epoch: 2


100%|██████████| 97/97 [00:17<00:00,  5.61it/s]


...

100%|██████████| 97/97 [00:14<00:00,  6.55it/s]



Train set: Average loss: 31791.6484, Accuracy: 63.87%
Val set: Average loss: 77674.0156, Accuracy: 54.74%

Classification report:
              precision    recall  f1-score   support

       sedan       0.66      0.57      0.61       418
         SUV       0.41      0.51      0.46       247

    accuracy                           0.55       665
   macro avg       0.54      0.54      0.53       665
weighted avg       0.57      0.55      0.55       665

Training is ended!


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


train_accuracy,▁▃▄▄▆▅█▇█▇
train_loss,█▄▃▂▃▂▁▂▂▁
val_accuracy,▅█▁▄▅▇▄▆█▅
val_loss,█▅▆▄▃▃▂▂▁▁
train_accuracy,0.63874
train_loss,31791.64844
val_accuracy,0.54737
val_loss,77674.01562


[OK] модель сохранена: ../data/models/BaseLine_FC_body_type.pt


Качество около рандомное, то есть не сильно лучше чем просто определять на рандом с 50% вероятностью (чуть меньше чем на 10% лучше)

#### Задача на определение марки

In [18]:
CFG.classes = all_brands

model_baseline_brand = FC_Net(task='brand')

In [21]:
summary(model=model_baseline_brand,
        input_size=(32, 3, 96, 160), # входной батч
        col_names=["input_size", "output_size", "num_params", "trainable"], # что хотим посмотреть
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
FC_Net                                   [32, 3, 96, 160]     [32, 8]              --                   True
├─Linear: 1-1                            [32, 46080]          [32, 1024]           47,186,944           True
├─Linear: 1-2                            [32, 1024]           [32, 512]            524,800              True
├─Linear: 1-3                            [32, 512]            [32, 128]            65,664               True
├─Linear: 1-4                            [32, 128]            [32, 8]              1,032                True
Total params: 47,778,440
Trainable params: 47,778,440
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 1.53
Input size (MB): 5.90
Forward/backward pass size (MB): 0.43
Params size (MB): 191.11
Estimated Total Size (MB): 197.44

In [22]:
main_kolesa(model_baseline_brand, train_df = train_df_brand, val_df = val_df_brand)
save_model_artifact(model_baseline_brand, 'BaseLine_FC_brand')


Epoch: 1


100%|██████████| 97/97 [00:15<00:00,  6.36it/s]



Train set: Average loss: 285470.5938, Accuracy: 13.89%
Val set: Average loss: 388377.4688, Accuracy: 12.18%

Classification report:
               precision    recall  f1-score   support

         Audi       0.08      0.10      0.09        73
          BMW       0.17      0.09      0.12        90
    Chevrolet       0.15      0.19      0.17        79
      Hyundai       0.11      0.13      0.12        85
          Kia       0.16      0.16      0.16        77
        Lexus       0.13      0.13      0.13        95
Mercedes-Benz       0.10      0.11      0.11        81
       Toyota       0.08      0.08      0.08        85

     accuracy                           0.12       665
    macro avg       0.12      0.12      0.12       665
 weighted avg       0.13      0.12      0.12       665


Epoch: 2


100%|██████████| 97/97 [00:14<00:00,  6.53it/s]



Train set: Average loss: 319479.6875, Accuracy: 13.63%
Val set: Average loss: 333627.9688, Accuracy: 11.28%

Classification report:
               precision    recall  f1-score   support

         Audi       0.07      0.07      0.07        73
          BMW       0.14      0.12      0.13        90
    Chevrolet       0.13      0.15      0.14        79
      Hyundai       0.11      0.12      0.12        85
          Kia       0.13      0.10      0.12        77
        Lexus       0.09      0.12      0.10        95
Mercedes-Benz       0.14      0.11      0.12        81
       Toyota       0.10      0.11      0.11        85

     accuracy                           0.11       665
    macro avg       0.11      0.11      0.11       665
 weighted avg       0.11      0.11      0.11       665


Epoch: 3


...

100%|██████████| 97/97 [00:15<00:00,  6.35it/s]



Train set: Average loss: 15272.5332, Accuracy: 16.79%
Val set: Average loss: 98214.9688, Accuracy: 14.14%

Classification report:
               precision    recall  f1-score   support

         Audi       0.09      0.10      0.09        73
          BMW       0.14      0.11      0.12        90
    Chevrolet       0.16      0.15      0.15        79
      Hyundai       0.11      0.11      0.11        85
          Kia       0.20      0.17      0.18        77
        Lexus       0.19      0.20      0.20        95
Mercedes-Benz       0.16      0.12      0.14        81
       Toyota       0.10      0.16      0.13        85

     accuracy                           0.14       665
    macro avg       0.14      0.14      0.14       665
 weighted avg       0.15      0.14      0.14       665

Training is ended!


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


train_accuracy,▁▁▃▅▆▇█▇▅▅
train_loss,▇█▅▆▅▃▃▂▂▁
val_accuracy,▃▁▇▂▃▄▇▅▇█
val_loss,█▇▅▅▄▃▃▂▁▁
train_accuracy,0.1679
train_loss,15272.5332
val_accuracy,0.14135
val_loss,98214.96875


[OK] модель сохранена: ../data/models/BaseLine_FC_brand.pt


Здесь уже чуть больше чем на 10% лучше чистого рандома, где угадать марку из 8 возможных - 12.5%. Но, в любом случае, такие результаты сильно плохи.

Попробуем далее сверточные сети.

## Простой CNN

In [19]:
class CNN_Net(nn.Module):
    def __init__(self, task='body_type'):
        super(CNN_Net, self).__init__()

        self.task = task

        self.conv1 = torch.nn.Conv2d(3, 16, 3, padding=1)
        self.act1  = torch.nn.ReLU()
        self.pool1 = torch.nn.MaxPool2d(2, 2) # 48 80

        self.conv2 = torch.nn.Conv2d(16, 32, 3, padding=1)
        self.act2  = torch.nn.ReLU()
        self.pool2 = torch.nn.MaxPool2d(2, 2) # 24 40

        self.conv3 = torch.nn.Conv2d(32, 64, 3, padding=1)
        self.act3  = torch.nn.ReLU()
        self.pool3 = torch.nn.MaxPool2d(2, 2) # 12 20

        self.fc1   = torch.nn.Linear(12 * 20 * 64, 256)
        self.act4  = torch.nn.Tanh()

        self.fc2   = torch.nn.Linear(256, 64)
        self.act5  = torch.nn.Tanh()
        
        if task == 'body_type':
            self.fc3   = torch.nn.Linear(64, 2)
        elif task == 'brand':
            self.fc3   = torch.nn.Linear(64, 8)

    def forward(self, x):
        x = self.conv1(x)
        x = self.act1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.act2(x)
        x = self.pool2(x)

        x = self.conv3(x)
        x = self.act3(x)
        x = self.pool3(x)

        x = x.view(x.size(0), x.size(1) * x.size(2) * x.size(3))
        x = self.fc1(x)
        x = self.act4(x)
        x = self.fc2(x)
        x = self.act5(x)
        x = self.fc3(x)

        return x

#### Задача на тип кузова

In [20]:
CFG.classes = all_body_types

model_cnn_1_bt = CNN_Net()

In [25]:
summary(model=model_cnn_1_bt,
        input_size=(32, 3, 96, 160), # входной батч
        col_names=["input_size", "output_size", "num_params", "trainable"], # что хотим посмотреть
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
CNN_Net                                  [32, 3, 96, 160]     [32, 2]              --                   True
├─Conv2d: 1-1                            [32, 3, 96, 160]     [32, 16, 96, 160]    448                  True
├─ReLU: 1-2                              [32, 16, 96, 160]    [32, 16, 96, 160]    --                   --
├─MaxPool2d: 1-3                         [32, 16, 96, 160]    [32, 16, 48, 80]     --                   --
├─Conv2d: 1-4                            [32, 16, 48, 80]     [32, 32, 48, 80]     4,640                True
├─ReLU: 1-5                              [32, 32, 48, 80]     [32, 32, 48, 80]     --                   --
├─MaxPool2d: 1-6                         [32, 32, 48, 80]     [32, 32, 24, 40]     --                   --
├─Conv2d: 1-7                            [32, 32, 24, 40]     [32, 64, 24, 40]     18,496               True
├─ReLU: 1-8           

In [26]:
main_kolesa(model_cnn_1_bt, train_df=train_df_bt, val_df=val_df_bt)
save_model_artifact(model_cnn_1_bt, 'Base_CNN_body_type')


Epoch: 1


100%|██████████| 97/97 [00:15<00:00,  6.46it/s]



Train set: Average loss: 0.7569, Accuracy: 62.20%
Val set: Average loss: 0.6641, Accuracy: 63.61%

Classification report:
              precision    recall  f1-score   support

       sedan       0.63      1.00      0.78       418
         SUV       1.00      0.02      0.04       247

    accuracy                           0.64       665
   macro avg       0.82      0.51      0.41       665
weighted avg       0.77      0.64      0.50       665


Epoch: 2


100%|██████████| 97/97 [00:14<00:00,  6.54it/s]



Train set: Average loss: 0.6975, Accuracy: 63.33%
Val set: Average loss: 0.6965, Accuracy: 57.29%

Classification report:
              precision    recall  f1-score   support

       sedan       0.75      0.49      0.59       418
         SUV       0.45      0.72      0.56       247

    accuracy                           0.57       665
   macro avg       0.60      0.60      0.57       665
weighted avg       0.64      0.57      0.58       665


Epoch: 3


...

100%|██████████| 97/97 [00:14<00:00,  6.70it/s]



Train set: Average loss: 0.6076, Accuracy: 72.32%
Val set: Average loss: 0.6844, Accuracy: 63.61%

Classification report:
              precision    recall  f1-score   support

       sedan       0.76      0.61      0.68       418
         SUV       0.51      0.68      0.58       247

    accuracy                           0.64       665
   macro avg       0.64      0.65      0.63       665
weighted avg       0.67      0.64      0.64       665

Training is ended!


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


train_accuracy,▁▂▂▄▄▅▅▇▇█
train_loss,█▆▄▄▃▃▁▃▁▂
val_accuracy,▆▁▅▆▇▇██▆▆
val_loss,▄█▃▂▁▂▁▁▃▆
train_accuracy,0.72317
train_loss,0.60758
val_accuracy,0.63609
val_loss,0.68437


[OK] модель сохранена: ../data/models/Base_CNN_body_type.pt


Уже получше, на +-20% лучшем чем FC, но пока что такая точность слишком низка для нашей задачи. 

#### Задача на определение марки

In [21]:
CFG.classes = all_brands

model_cnn_1_brand = CNN_Net(task='brand')

In [28]:
summary(model=model_cnn_1_brand,
        input_size=(32, 3, 96, 160), # входной батч
        col_names=["input_size", "output_size", "num_params", "trainable"], # что хотим посмотреть
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
CNN_Net                                  [32, 3, 96, 160]     [32, 8]              --                   True
├─Conv2d: 1-1                            [32, 3, 96, 160]     [32, 16, 96, 160]    448                  True
├─ReLU: 1-2                              [32, 16, 96, 160]    [32, 16, 96, 160]    --                   --
├─MaxPool2d: 1-3                         [32, 16, 96, 160]    [32, 16, 48, 80]     --                   --
├─Conv2d: 1-4                            [32, 16, 48, 80]     [32, 32, 48, 80]     4,640                True
├─ReLU: 1-5                              [32, 32, 48, 80]     [32, 32, 48, 80]     --                   --
├─MaxPool2d: 1-6                         [32, 32, 48, 80]     [32, 32, 24, 40]     --                   --
├─Conv2d: 1-7                            [32, 32, 24, 40]     [32, 64, 24, 40]     18,496               True
├─ReLU: 1-8           

In [29]:
main_kolesa(model_cnn_1_brand, train_df=train_df_brand, val_df=val_df_brand)
save_model_artifact(model_cnn_1_brand, 'Base_CNN_brand')


Epoch: 1


100%|██████████| 97/97 [00:14<00:00,  6.65it/s]



Train set: Average loss: 2.0729, Accuracy: 14.24%


/Users/ksa/smadimo/dev_gp5/gp_5_DL/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/ksa/smadimo/dev_gp5/gp_5_DL/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/ksa/smadimo/dev_gp5/gp_5_DL/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

Val set: Average loss: 2.0768, Accuracy: 14.14%

Classification report:
               precision    recall  f1-score   support

         Audi       0.00      0.00      0.00        73
          BMW       0.11      0.33      0.17        90
    Chevrolet       0.21      0.23      0.22        79
      Hyundai       0.00      0.00      0.00        85
          Kia       0.23      0.17      0.20        77
        Lexus       0.17      0.24      0.20        95
Mercedes-Benz       0.04      0.01      0.02        81
       Toyota       0.09      0.11      0.10        85

     accuracy                           0.14       665
    macro avg       0.11      0.14      0.11       665
 weighted avg       0.11      0.14      0.11       665


Epoch: 2


...


Train set: Average loss: 1.5459, Accuracy: 33.23%
Val set: Average loss: 2.1410, Accuracy: 19.10%

Classification report:
               precision    recall  f1-score   support

         Audi       0.17      0.22      0.19        73
          BMW       0.22      0.48      0.30        90
    Chevrolet       0.50      0.16      0.25        79
      Hyundai       0.10      0.01      0.02        85
          Kia       0.22      0.18      0.20        77
        Lexus       0.19      0.08      0.12        95
Mercedes-Benz       0.15      0.26      0.19        81
       Toyota       0.12      0.13      0.12        85

     accuracy                           0.19       665
    macro avg       0.21      0.19      0.17       665
 weighted avg       0.21      0.19      0.17       665

Training is ended!


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


wandb: WARNING Artifact "Base_CNN_brand" already exists with the same content. No new version will be created.


train_accuracy,▁▂▃▄▄▅▅▆▇█
train_loss,██▇▇▇▆▅▅▃▁
val_accuracy,▁▄▃▅▅▅▅█▆█
val_loss,▂▂▂▂▁▂▄▅█▇
train_accuracy,0.33226
train_loss,1.54587
val_accuracy,0.19098
val_loss,2.14097


[OK] модель сохранена: ../data/models/Base_CNN_brand.pt


Прирост выше, чем для задачи на кузов, уже чуть более 30%, хотя это также все еще плохо, с какими-то марками (как к примеру Hyundai, где по ней precision и recall по нулям) большие проблемы

## CNN с регуляризацией

Теперь попробуем добавить регуляризацию, а именно известные нам BatchNorm и Dropout.

Также поменяем функции активации https://deepmachinelearning.ru/docs/Neural-networks/MLP/Activation-functions. В частности попробуем LeakyReLU.

И наконец добавим побольше сверточных слоев.

In [22]:
class CNNReg_Net(nn.Module):
    def __init__(self, task='body_type'):
        super(CNNReg_Net, self).__init__()

        self.task = task

        self.conv3 = torch.nn.Conv2d(3, 32, 3, padding=1)
        self.bn3   = torch.nn.BatchNorm2d(32)
        self.act3  = torch.nn.LeakyReLU()

        self.conv4 = torch.nn.Conv2d(32, 32, 3, padding=1)
        self.bn4   = torch.nn.BatchNorm2d(32)
        self.act4  = torch.nn.LeakyReLU()
        self.pool2 = torch.nn.MaxPool2d(2, 2) # 24 40

        self.conv5 = torch.nn.Conv2d(32, 64, 3, padding=1)
        self.bn5   = torch.nn.BatchNorm2d(64)
        self.act5  = torch.nn.LeakyReLU()

        self.conv6 = torch.nn.Conv2d(64, 64, 3, padding=1)
        self.bn6   = torch.nn.BatchNorm2d(64)
        self.act6  = torch.nn.LeakyReLU()
        self.pool3 = torch.nn.MaxPool2d(2, 2) # 12 20

        self.drop1 = torch.nn.Dropout2d(0.10)

        self.conv7 = torch.nn.Conv2d(64, 128, 3, padding=1)
        self.bn7   = torch.nn.BatchNorm2d(128)
        self.act7  = torch.nn.LeakyReLU()
        self.drop2 = torch.nn.Dropout2d(0.15)

        self.conv8 = torch.nn.Conv2d(128, 128, 3, padding=1)
        self.bn8   = torch.nn.BatchNorm2d(128)
        self.act8  = torch.nn.LeakyReLU()
        self.pool4 = torch.nn.MaxPool2d(2, 2)
        self.drop3 = torch.nn.Dropout2d(0.2)

        self.conv9 = torch.nn.Conv2d(128, 256, 3, padding=1)
        self.bn9   = torch.nn.BatchNorm2d(256)
        self.act9  = torch.nn.LeakyReLU()
        self.pool5 = torch.nn.MaxPool2d(2, 2)
        self.drop4 = torch.nn.Dropout2d(0.25)

        self.fc1   = torch.nn.Linear(6 * 10 * 256, 256)
        self.bn10  = torch.nn.BatchNorm1d(256)
        self.act10 = torch.nn.LeakyReLU()
        self.drop5 = torch.nn.Dropout(0.3)

        self.fc2   = torch.nn.Linear(256, 64)
        self.act11 = torch.nn.LeakyReLU()
        self.drop6 = torch.nn.Dropout(0.2)

        if task == 'body_type':
            self.fc3 = torch.nn.Linear(64, 2)
        elif task == 'brand':
            self.fc3 = torch.nn.Linear(64, 8)
            

    def forward(self, x):
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.act3(x)

        x = self.conv4(x)
        x = self.bn4(x)
        x = self.act4(x)
        x = self.pool2(x)

        x = self.conv5(x)
        x = self.bn5(x)
        x = self.act5(x)

        x = self.conv6(x)
        x = self.bn6(x)
        x = self.act6(x)
        x = self.pool3(x)
        x = self.drop1(x)

        x = self.conv7(x)
        x = self.bn7(x)
        x = self.act7(x)
        x = self.drop2(x)

        x = self.conv8(x)
        x = self.bn8(x)
        x = self.act8(x)
        x = self.pool4(x)
        x = self.drop3(x)

        x = self.conv9(x)
        x = self.bn9(x)
        x = self.act9(x)
        x = self.pool5(x)
        x = self.drop4(x)

        x = x.view(x.size(0), x.size(1) * x.size(2) * x.size(3))
        x = self.fc1(x)
        x = self.bn10(x)
        x = self.act10(x)
        x = self.drop5(x)

        x = self.fc2(x)
        x = self.act11(x)
        x = self.drop6(x)

        x = self.fc3(x)

        return x

На этой модели попробуем побольше эпох

In [23]:
CFG.num_epochs = 20

#### Задача на тип кузова

In [24]:
CFG.classes = all_body_types

model_cnn_2_bt = CNNReg_Net()

In [33]:
summary(model=model_cnn_2_bt,
        input_size=(32, 3, 96, 160), # входной батч
        col_names=["input_size", "output_size", "num_params", "trainable"], # что хотим посмотреть
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
CNNReg_Net                               [32, 3, 96, 160]     [32, 2]              --                   True
├─Conv2d: 1-1                            [32, 3, 96, 160]     [32, 32, 96, 160]    896                  True
├─BatchNorm2d: 1-2                       [32, 32, 96, 160]    [32, 32, 96, 160]    64                   True
├─LeakyReLU: 1-3                         [32, 32, 96, 160]    [32, 32, 96, 160]    --                   --
├─Conv2d: 1-4                            [32, 32, 96, 160]    [32, 32, 96, 160]    9,248                True
├─BatchNorm2d: 1-5                       [32, 32, 96, 160]    [32, 32, 96, 160]    64                   True
├─LeakyReLU: 1-6                         [32, 32, 96, 160]    [32, 32, 96, 160]    --                   --
├─MaxPool2d: 1-7                         [32, 32, 96, 160]    [32, 32, 48, 80]     --                   --
├─Conv2d: 1-8       

In [34]:
main_kolesa(model_cnn_2_bt, train_df=train_df_bt, val_df=val_df_bt)
save_model_artifact(model_cnn_2_bt, 'Reg_CNN_body_type')


Epoch: 1


100%|██████████| 97/97 [00:18<00:00,  5.37it/s]



Train set: Average loss: 0.8177, Accuracy: 61.20%
Val set: Average loss: 0.6174, Accuracy: 65.86%

Classification report:
              precision    recall  f1-score   support

       sedan       0.67      0.92      0.77       418
         SUV       0.61      0.22      0.32       247

    accuracy                           0.66       665
   macro avg       0.64      0.57      0.55       665
weighted avg       0.65      0.66      0.60       665


Epoch: 2


100%|██████████| 97/97 [00:17<00:00,  5.50it/s]



Train set: Average loss: 0.7784, Accuracy: 63.81%
Val set: Average loss: 0.6206, Accuracy: 65.86%

Classification report:
              precision    recall  f1-score   support

       sedan       0.70      0.79      0.74       418
         SUV       0.55      0.44      0.49       247

    accuracy                           0.66       665
   macro avg       0.63      0.61      0.62       665
weighted avg       0.65      0.66      0.65       665


Epoch: 3


...

100%|██████████| 97/97 [00:17<00:00,  5.46it/s]



Train set: Average loss: 0.4856, Accuracy: 79.28%
Val set: Average loss: 0.6110, Accuracy: 67.82%

Classification report:
              precision    recall  f1-score   support

       sedan       0.83      0.62      0.71       418
         SUV       0.55      0.78      0.64       247

    accuracy                           0.68       665
   macro avg       0.69      0.70      0.68       665
weighted avg       0.72      0.68      0.68       665

Training is ended!


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


train_accuracy,▁▂▃▃▃▄▅▅▅▆▆▆▆▇▇▇████
train_loss,█▇▆▇▆▇▅▅▅▆▃▄▄▇▃▆▆▄▄▁
val_accuracy,▃▃▅▁▅▆▆▁▃▅███▆▅█▇█▅▄
val_loss,▆▆▅▆▄▃▃█▇▅▁▃▁▅▇▁▅▂▆▆
train_accuracy,0.79278
train_loss,0.48561
val_accuracy,0.6782
val_loss,0.611


[OK] модель сохранена: ../data/models/Reg_CNN_body_type.pt


73.38% accuracy на лучшей эпохе это уже более-менее, хотя пока что с SUV, которых меньше, все еще сложнее, из-за этого доверять этой модели сложно.

#### Задача на определение марки

In [25]:
CFG.classes = all_brands

model_cnn_2_brand = CNNReg_Net(task='brand')

In [36]:
summary(model=model_cnn_2_brand,
        input_size=(32, 3, 96, 160), # входной батч
        col_names=["input_size", "output_size", "num_params", "trainable"], # что хотим посмотреть
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
CNNReg_Net                               [32, 3, 96, 160]     [32, 8]              --                   True
├─Conv2d: 1-1                            [32, 3, 96, 160]     [32, 32, 96, 160]    896                  True
├─BatchNorm2d: 1-2                       [32, 32, 96, 160]    [32, 32, 96, 160]    64                   True
├─LeakyReLU: 1-3                         [32, 32, 96, 160]    [32, 32, 96, 160]    --                   --
├─Conv2d: 1-4                            [32, 32, 96, 160]    [32, 32, 96, 160]    9,248                True
├─BatchNorm2d: 1-5                       [32, 32, 96, 160]    [32, 32, 96, 160]    64                   True
├─LeakyReLU: 1-6                         [32, 32, 96, 160]    [32, 32, 96, 160]    --                   --
├─MaxPool2d: 1-7                         [32, 32, 96, 160]    [32, 32, 48, 80]     --                   --
├─Conv2d: 1-8       

In [37]:
main_kolesa(model_cnn_2_brand, train_df=train_df_brand, val_df=val_df_brand)
save_model_artifact(model_cnn_2_brand, 'Reg_CNN_brand')


Epoch: 1


100%|██████████| 97/97 [00:17<00:00,  5.66it/s]



Train set: Average loss: 2.0438, Accuracy: 15.21%
Val set: Average loss: 2.0703, Accuracy: 14.74%

Classification report:
               precision    recall  f1-score   support

         Audi       0.14      0.10      0.11        73
          BMW       0.13      0.29      0.18        90
    Chevrolet       0.16      0.52      0.24        79
      Hyundai       0.00      0.00      0.00        85
          Kia       0.20      0.18      0.19        77
        Lexus       0.06      0.02      0.03        95
Mercedes-Benz       0.29      0.05      0.08        81
       Toyota       0.10      0.05      0.06        85

     accuracy                           0.15       665
    macro avg       0.13      0.15      0.11       665
 weighted avg       0.13      0.15      0.11       665


Epoch: 2


100%|██████████| 97/97 [00:17<00:00,  5.50it/s]



Train set: Average loss: 2.0708, Accuracy: 17.60%
Val set: Average loss: 2.0554, Accuracy: 16.39%

Classification report:
               precision    recall  f1-score   support

         Audi       0.00      0.00      0.00        73
          BMW       0.13      0.22      0.17        90
    Chevrolet       0.18      0.43      0.25        79
      Hyundai       0.00      0.00      0.00        85
          Kia       0.29      0.16      0.20        77
        Lexus       0.16      0.22      0.19        95
Mercedes-Benz       0.16      0.04      0.06        81
       Toyota       0.15      0.22      0.18        85

     accuracy                           0.16       665
    macro avg       0.13      0.16      0.13       665
 weighted avg       0.13      0.16      0.13       665


Epoch: 3


...

100%|██████████| 97/97 [00:17<00:00,  5.48it/s]



Train set: Average loss: 1.5164, Accuracy: 36.58%
Val set: Average loss: 1.9102, Accuracy: 28.87%

Classification report:
               precision    recall  f1-score   support

         Audi       0.26      0.26      0.26        73
          BMW       0.29      0.73      0.41        90
    Chevrolet       0.47      0.47      0.47        79
      Hyundai       0.37      0.12      0.18        85
          Kia       0.28      0.19      0.23        77
        Lexus       0.22      0.15      0.18        95
Mercedes-Benz       0.28      0.20      0.23        81
       Toyota       0.19      0.18      0.18        85

     accuracy                           0.29       665
    macro avg       0.29      0.29      0.27       665
 weighted avg       0.29      0.29      0.27       665

Training is ended!


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


train_accuracy,▁▂▂▃▃▃▄▄▅▄▅▆▆▆▇▇▇███
train_loss,██▇▇▇▆▅▅▄▄▄▃▃▃▄▃▃▄▁▁
val_accuracy,▁▂▂▃▄▃▄▄▅▇▆▆▇▆██▇▇▇█
val_loss,█▇▇▆▅▅▄▃▄▂▄▃▂▃▂▁▁▁▂▂
train_accuracy,0.36578
train_loss,1.51641
val_accuracy,0.28872
val_loss,1.91023


[OK] модель сохранена: ../data/models/Reg_CNN_brand.pt


29.17% на лучшей эпохе это уже в два раза лучше чем на baseline. Это хороший буст, но пока это все же плохо.


Вернем дефолтное количество эпох

In [26]:
CFG.num_epochs = 10

Далее мы уже будем брать за основу предобученные модели и пытаться использовать FineTuning. Посмотрим далее помогло ли это нам в наших задачах.

## FineTuned AlexNet

Далее попробуем в качестве другого подхода взять уже ранее предобученную модель. Возьмем AlexNet

Из-за проблем при запуске AlexNet с MacBook используя MPS - мы вынесли эти запуски в отдельный ноутбук `alexnet.ipynb`, где делали запуски в Colab.

## ResNet18

Возьмём более современную **ResNet18**.

Ее преимущество - остаточные связи, которые позволяют обучать глубокие сети без затухания градиента.

Дообучаем так же - через наш `main_kolesa`, заменив только финальный полносвязный слой.

In [27]:
from torchvision.models import resnet18
from torchvision.models import ResNet18_Weights

#### Задача на тип кузова

In [28]:
CFG.classes = all_body_types
num_out_features = 2

model_resnet_bt = resnet18(weights=ResNet18_Weights.DEFAULT)
model_resnet_bt.fc = torch.nn.Linear(model_resnet_bt.fc.in_features, num_out_features)

In [41]:
summary(model=model_resnet_bt,
        input_size=(32, 3, 96, 160),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
ResNet                                   [32, 3, 96, 160]     [32, 2]              --                   True
├─Conv2d: 1-1                            [32, 3, 96, 160]     [32, 64, 48, 80]     9,408                True
├─BatchNorm2d: 1-2                       [32, 64, 48, 80]     [32, 64, 48, 80]     128                  True
├─ReLU: 1-3                              [32, 64, 48, 80]     [32, 64, 48, 80]     --                   --
├─MaxPool2d: 1-4                         [32, 64, 48, 80]     [32, 64, 24, 40]     --                   --
├─Sequential: 1-5                        [32, 64, 24, 40]     [32, 64, 24, 40]     --                   True
│    └─BasicBlock: 2-1                   [32, 64, 24, 40]     [32, 64, 24, 40]     --                   True
│    │    └─Conv2d: 3-1                  [32, 64, 24, 40]     [32, 64, 24, 40]     36,864               True
│    │    └─BatchN

In [42]:
main_kolesa(model_resnet_bt, train_df=train_df_bt, val_df=val_df_bt)
save_model_artifact(model_resnet_bt, 'ResNet18_finetune_body_type')


Epoch: 1


100%|██████████| 97/97 [00:17<00:00,  5.58it/s]



Train set: Average loss: 0.5798, Accuracy: 72.64%
Val set: Average loss: 0.3939, Accuracy: 83.91%

Classification report:
              precision    recall  f1-score   support

       sedan       0.88      0.86      0.87       418
         SUV       0.77      0.80      0.79       247

    accuracy                           0.84       665
   macro avg       0.83      0.83      0.83       665
weighted avg       0.84      0.84      0.84       665


Epoch: 2


100%|██████████| 97/97 [00:16<00:00,  5.77it/s]



Train set: Average loss: 0.5523, Accuracy: 84.05%
Val set: Average loss: 0.3788, Accuracy: 84.21%

Classification report:
              precision    recall  f1-score   support

       sedan       0.82      0.96      0.88       418
         SUV       0.91      0.64      0.75       247

    accuracy                           0.84       665
   macro avg       0.86      0.80      0.82       665
weighted avg       0.85      0.84      0.83       665


Epoch: 3


...

100%|██████████| 97/97 [00:16<00:00,  5.83it/s]



Train set: Average loss: 0.0986, Accuracy: 96.71%
Val set: Average loss: 0.2851, Accuracy: 87.52%

Classification report:
              precision    recall  f1-score   support

       sedan       0.89      0.91      0.90       418
         SUV       0.84      0.82      0.83       247

    accuracy                           0.88       665
   macro avg       0.87      0.86      0.87       665
weighted avg       0.87      0.88      0.87       665

Training is ended!


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


train_accuracy,▁▄▆▆▇▇▇███
train_loss,██▅▂▅▃▃▁▂▂
val_accuracy,▄▄▂▆▁▆▄█▂▇
val_loss,▆▆▇▆█▆▅▁▅▃
train_accuracy,0.96713
train_loss,0.09857
val_accuracy,0.87519
val_loss,0.28507


[OK] модель сохранена: ../data/models/ResNet18_finetune_body_type.pt


88.57% accuracy на лучшей эпохе - отличный результат, относительно того, что мы видели. При этом precision для SUV даже выше, хотя recall похуже. То есть модель хорошо попадает в SUV, но меньше их распознает в датасете.

#### Задача на определение марки

In [29]:
CFG.classes = all_brands
num_out_features = 8

model_resnet_brand = resnet18(weights=ResNet18_Weights.DEFAULT)
model_resnet_brand.fc = torch.nn.Linear(model_resnet_brand.fc.in_features, num_out_features)

In [44]:
summary(model=model_resnet_brand,
        input_size=(32, 3, 96, 160),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
ResNet                                   [32, 3, 96, 160]     [32, 8]              --                   True
├─Conv2d: 1-1                            [32, 3, 96, 160]     [32, 64, 48, 80]     9,408                True
├─BatchNorm2d: 1-2                       [32, 64, 48, 80]     [32, 64, 48, 80]     128                  True
├─ReLU: 1-3                              [32, 64, 48, 80]     [32, 64, 48, 80]     --                   --
├─MaxPool2d: 1-4                         [32, 64, 48, 80]     [32, 64, 24, 40]     --                   --
├─Sequential: 1-5                        [32, 64, 24, 40]     [32, 64, 24, 40]     --                   True
│    └─BasicBlock: 2-1                   [32, 64, 24, 40]     [32, 64, 24, 40]     --                   True
│    │    └─Conv2d: 3-1                  [32, 64, 24, 40]     [32, 64, 24, 40]     36,864               True
│    │    └─BatchN

In [45]:
main_kolesa(model_resnet_brand, train_df=train_df_brand, val_df=val_df_brand)
save_model_artifact(model_resnet_brand, 'ResNet18_finetune_brand')


Epoch: 1


100%|██████████| 97/97 [00:17<00:00,  5.61it/s]



Train set: Average loss: 1.6750, Accuracy: 26.17%
Val set: Average loss: 1.5889, Accuracy: 38.95%

Classification report:
               precision    recall  f1-score   support

         Audi       0.87      0.36      0.50        73
          BMW       0.81      0.51      0.63        90
    Chevrolet       0.59      0.43      0.50        79
      Hyundai       0.37      0.22      0.28        85
          Kia       0.29      0.42      0.34        77
        Lexus       0.27      0.54      0.36        95
Mercedes-Benz       0.31      0.58      0.40        81
       Toyota       0.25      0.05      0.08        85

     accuracy                           0.39       665
    macro avg       0.47      0.39      0.39       665
 weighted avg       0.46      0.39      0.38       665


Epoch: 2


100%|██████████| 97/97 [00:16<00:00,  5.74it/s]



Train set: Average loss: 1.0810, Accuracy: 50.15%
Val set: Average loss: 1.5561, Accuracy: 49.62%

Classification report:
               precision    recall  f1-score   support

         Audi       0.58      0.58      0.58        73
          BMW       0.72      0.76      0.74        90
    Chevrolet       0.58      0.57      0.57        79
      Hyundai       0.39      0.38      0.38        85
          Kia       0.36      0.66      0.47        77
        Lexus       0.54      0.43      0.48        95
Mercedes-Benz       0.50      0.36      0.42        81
       Toyota       0.36      0.26      0.30        85

     accuracy                           0.50       665
    macro avg       0.50      0.50      0.49       665
 weighted avg       0.50      0.50      0.49       665


Epoch: 3


...

100%|██████████| 97/97 [00:18<00:00,  5.39it/s]



Train set: Average loss: 0.1054, Accuracy: 91.65%
Val set: Average loss: 2.0116, Accuracy: 60.90%

Classification report:
               precision    recall  f1-score   support

         Audi       0.76      0.56      0.65        73
          BMW       0.84      0.73      0.78        90
    Chevrolet       0.61      0.87      0.72        79
      Hyundai       0.44      0.65      0.52        85
          Kia       0.67      0.42      0.51        77
        Lexus       0.64      0.60      0.62        95
Mercedes-Benz       0.53      0.67      0.59        81
       Toyota       0.56      0.36      0.44        85

     accuracy                           0.61       665
    macro avg       0.63      0.61      0.60       665
 weighted avg       0.63      0.61      0.61       665

Training is ended!


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


train_accuracy,▁▄▅▆▇▇▇███
train_loss,█▅▃▄▂▂▁▁▃▁
val_accuracy,▁▄▅▅▆▆▆▆▇█
val_loss,▃▂▁▄▃▅▅█▇▇
train_accuracy,0.91653
train_loss,0.10538
val_accuracy,0.60902
val_loss,2.01157


[OK] модель сохранена: ../data/models/ResNet18_finetune_brand.pt


60.90% accuracy на лучше эпохе пока лучшее, что видели. С какими-то марками, при этом, все сильно лучше, к примеру, у BMW. А вот явно хуже получается с Toyot'ой. При этом модель сильно переобучилась, а до этого модели даже на train работали плохо.

### ResNet с регуляризацией

Попробуем побороться с переобучением, за счет изменения основной main функции, добавления dropout в сам ResNet + обновим часть конфига. И попробуем взять больще эпохж

In [30]:
def main_kolesa_new(model, train_df, val_df):

    if CFG.wandb:
        wandb.init(project=CFG.project, entity=CFG.entity, reinit=True, config=class2dict(CFG))
        # логируем архитектуру модели
        # https://docs.wandb.ai/guides/track/config
        wandb.config.update({'model': str(model)})

    seed_everything(CFG.seed)

    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')

    kwargs = {'num_workers': CFG.num_workers, 'pin_memory': torch.cuda.is_available()}

    # Попробуем картинки побольше
    # будем использовать различные методы аугментации https://docs.pytorch.org/vision/0.13/transforms.html
    train_transform = transforms.Compose([
        transforms.Resize((320, 512)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225)
        )
        ])

    test_transform = transforms.Compose([
        transforms.Resize((320, 512)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225)
        )
        ])

    train_ds = KolesaDataset(train_df, transform=train_transform)
    val_ds = KolesaDataset(val_df, transform=test_transform)
    # test_ds = KolesaDataset(test_df, transform=test_transform)

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=CFG.train_batch_size, **kwargs)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=CFG.test_batch_size, **kwargs)
    # test_loader = torch.utils.data.DataLoader(test_ds, batch_size=CFG.test_batch_size, **kwargs)

    model = model.to(device)

    if CFG.wandb:
        wandb.watch(model, log='all')

    # https://habr.com/ru/articles/988198/
    # добавим l2 регуляризацию (weight_decay)
    optimizer = optim.Adam(model.parameters(),
                          lr=CFG.lr, weight_decay=3e-4)

    # добавим небольшой коэф. сглаживания, что может помочь с переобучением
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

    for epoch in range(1, CFG.num_epochs + 1):
        print('\nEpoch:', epoch)
        train(model, device, train_loader, optimizer, criterion, epoch, CFG.wandb)
        val(model, device, val_loader, criterion, CFG.wandb)

    print('Training is ended!')

Изменим learning rate и возьмем побольше эпох

In [31]:
CFG.num_epochs = 15
CFG.lr = 3e-5

### Задача на тип кузова

In [32]:
CFG.classes = all_body_types
num_out_features = 2
CFG.test_batch_size = 64 # иначе не хватает памяти с дефолтным 512

# https://discuss.pytorch.org/t/resnet-last-layer-modification/33530
model_resnet_bt_reg = resnet18(weights=ResNet18_Weights.DEFAULT)
model_resnet_bt_reg.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(model_resnet_bt_reg.fc.in_features, num_out_features)
) 

In [49]:
summary(model=model_resnet_bt_reg,
        input_size=(32, 3, 320, 512),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
ResNet                                   [32, 3, 320, 512]    [32, 2]              --                   True
├─Conv2d: 1-1                            [32, 3, 320, 512]    [32, 64, 160, 256]   9,408                True
├─BatchNorm2d: 1-2                       [32, 64, 160, 256]   [32, 64, 160, 256]   128                  True
├─ReLU: 1-3                              [32, 64, 160, 256]   [32, 64, 160, 256]   --                   --
├─MaxPool2d: 1-4                         [32, 64, 160, 256]   [32, 64, 80, 128]    --                   --
├─Sequential: 1-5                        [32, 64, 80, 128]    [32, 64, 80, 128]    --                   True
│    └─BasicBlock: 2-1                   [32, 64, 80, 128]    [32, 64, 80, 128]    --                   True
│    │    └─Conv2d: 3-1                  [32, 64, 80, 128]    [32, 64, 80, 128]    36,864               True
│    │    └─BatchN

In [50]:
main_kolesa_new(model_resnet_bt_reg, train_df=train_df_bt, val_df=val_df_bt)
save_model_artifact(model_resnet_bt_reg, 'ResNet18_reg_finetune_bt')


Epoch: 1


100%|██████████| 97/97 [01:07<00:00,  1.44it/s]



Train set: Average loss: 0.5697, Accuracy: 71.12%
Val set: Average loss: 0.3458, Accuracy: 85.56%

Classification report:
              precision    recall  f1-score   support

       sedan       0.85      0.94      0.89       418
         SUV       0.87      0.72      0.79       247

    accuracy                           0.86       665
   macro avg       0.86      0.83      0.84       665
weighted avg       0.86      0.86      0.85       665


Epoch: 2


100%|██████████| 97/97 [01:04<00:00,  1.50it/s]



Train set: Average loss: 0.5450, Accuracy: 86.98%
Val set: Average loss: 0.2402, Accuracy: 88.27%

Classification report:
              precision    recall  f1-score   support

       sedan       0.88      0.94      0.91       418
         SUV       0.89      0.78      0.83       247

    accuracy                           0.88       665
   macro avg       0.88      0.86      0.87       665
weighted avg       0.88      0.88      0.88       665


Epoch: 3


...

100%|██████████| 97/97 [01:02<00:00,  1.54it/s]



Train set: Average loss: 0.1580, Accuracy: 99.94%
Val set: Average loss: 0.2077, Accuracy: 93.68%

Classification report:
              precision    recall  f1-score   support

       sedan       0.93      0.98      0.95       418
         SUV       0.96      0.87      0.91       247

    accuracy                           0.94       665
   macro avg       0.94      0.92      0.93       665
weighted avg       0.94      0.94      0.94       665

Training is ended!


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


train_accuracy,▁▅▆▇▇██████████
train_loss,██▅▄▃▂▃▂▂▂▁▂▁▁▁
val_accuracy,▁▃▄▆▆▇▇▇▇██▇███
val_loss,█▃▃▂▂▁▂▁▁▂▂▄▂▂▁
train_accuracy,0.99936
train_loss,0.15799
val_accuracy,0.93684
val_loss,0.20767


[OK] модель сохранена: ../data/models/ResNet18_reg_finetune_bt.pt


94.29% - очень хороший результат. Все также у седанов выше recall, а у SUV precision.

### Задача на марки

In [33]:
CFG.classes = all_brands
num_out_features = 8
# https://discuss.pytorch.org/t/resnet-last-layer-modification/33530
model_resnet_brand_reg = resnet18(weights=ResNet18_Weights.DEFAULT)
model_resnet_brand_reg.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(model_resnet_brand.fc.in_features, num_out_features)
) 

In [52]:
summary(model=model_resnet_brand_reg,
        input_size=(32, 3, 96, 160),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20
)

Layer (type:depth-idx)                   Input Shape          Output Shape         Param #              Trainable
ResNet                                   [32, 3, 96, 160]     [32, 8]              --                   True
├─Conv2d: 1-1                            [32, 3, 96, 160]     [32, 64, 48, 80]     9,408                True
├─BatchNorm2d: 1-2                       [32, 64, 48, 80]     [32, 64, 48, 80]     128                  True
├─ReLU: 1-3                              [32, 64, 48, 80]     [32, 64, 48, 80]     --                   --
├─MaxPool2d: 1-4                         [32, 64, 48, 80]     [32, 64, 24, 40]     --                   --
├─Sequential: 1-5                        [32, 64, 24, 40]     [32, 64, 24, 40]     --                   True
│    └─BasicBlock: 2-1                   [32, 64, 24, 40]     [32, 64, 24, 40]     --                   True
│    │    └─Conv2d: 3-1                  [32, 64, 24, 40]     [32, 64, 24, 40]     36,864               True
│    │    └─BatchN

In [53]:
main_kolesa_new(model_resnet_brand_reg, train_df=train_df_brand, val_df=val_df_brand)
save_model_artifact(model_resnet_brand_reg, 'ResNet18_reg_finetune_brand')


Epoch: 1


100%|██████████| 97/97 [01:02<00:00,  1.54it/s]



Train set: Average loss: 1.9662, Accuracy: 20.50%
Val set: Average loss: 1.9145, Accuracy: 38.80%

Classification report:
               precision    recall  f1-score   support

         Audi       0.51      0.32      0.39        73
          BMW       0.45      0.84      0.59        90
    Chevrolet       0.33      0.71      0.45        79
      Hyundai       0.43      0.11      0.17        85
          Kia       0.46      0.14      0.22        77
        Lexus       0.33      0.40      0.36        95
Mercedes-Benz       0.38      0.51      0.44        81
       Toyota       0.27      0.05      0.08        85

     accuracy                           0.39       665
    macro avg       0.40      0.38      0.34       665
 weighted avg       0.39      0.39      0.34       665


Epoch: 2


100%|██████████| 97/97 [01:02<00:00,  1.56it/s]



Train set: Average loss: 1.4085, Accuracy: 43.06%
Val set: Average loss: 1.4325, Accuracy: 56.24%

Classification report:
               precision    recall  f1-score   support

         Audi       0.68      0.68      0.68        73
          BMW       0.76      0.82      0.79        90
    Chevrolet       0.54      0.76      0.63        79
      Hyundai       0.55      0.49      0.52        85
          Kia       0.56      0.32      0.41        77
        Lexus       0.36      0.61      0.45        95
Mercedes-Benz       0.72      0.63      0.67        81
       Toyota       0.48      0.16      0.25        85

     accuracy                           0.56       665
    macro avg       0.58      0.56      0.55       665
 weighted avg       0.58      0.56      0.55       665


Epoch: 3


...

100%|██████████| 97/97 [01:03<00:00,  1.53it/s]



Train set: Average loss: 0.3223, Accuracy: 99.16%
Val set: Average loss: 0.6292, Accuracy: 87.07%

Classification report:
               precision    recall  f1-score   support

         Audi       0.90      0.82      0.86        73
          BMW       0.89      0.91      0.90        90
    Chevrolet       0.84      0.94      0.89        79
      Hyundai       0.84      0.89      0.87        85
          Kia       0.93      0.87      0.90        77
        Lexus       0.86      0.83      0.84        95
Mercedes-Benz       0.84      0.91      0.88        81
       Toyota       0.88      0.79      0.83        85

     accuracy                           0.87       665
    macro avg       0.87      0.87      0.87       665
 weighted avg       0.87      0.87      0.87       665

Training is ended!


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


train_accuracy,▁▃▅▆▆▇▇▇▇██████
train_loss,█▆▄▃▃▂▂▂▂▁▁▂▁▁▁
val_accuracy,▁▄▆▇▇▇█████████
val_loss,█▅▄▃▂▂▂▂▁▁▁▁▁▁▁
train_accuracy,0.99162
train_loss,0.32234
val_accuracy,0.87068
val_loss,0.62925


[OK] модель сохранена: ../data/models/ResNet18_reg_finetune_brand.pt


87.07% - догнали результат ResNet на типах кузова на задаче, где мы стартовали с 14% accuracy - это отличный прогресс для такой сложной задачи. И уже, к примеру, Toyota не выглядит такой проблемной, в целом, у всех марок примерно схожие результаты по метрикам, так что это объективная крепкая модель + показывает эту огромную разницу с полностью самостоятельными моделями. Вохможно, было бы у нас больше данных, то на начальных моделях были бы результаты лучше, но мы постарались в рамках наших ограничений выжать максимум. 

## Сравнение и выводы

Теперь для всех сохраненных моделей сделаем прогоны на test и сохраним результаты. Полное сранение сделаем уже в другом ноутбуке.

Будем брать сохраненные веса с .pt файлов

In [34]:
# https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html
def load_model(model, model_name):
    path = base_dir / 'models' / f'{model_name}.pt'
    model.load_state_dict(torch.load(path, weights_only=True))
    return model

model_baseline_bt = load_model(model_baseline_bt, 'BaseLine_FC_body_type')
model_cnn_1_bt = load_model(model_cnn_1_bt, 'Base_CNN_body_type')
model_cnn_2_bt = load_model(model_cnn_2_bt, 'Reg_CNN_body_type')
model_resnet_bt = load_model(model_resnet_bt, 'ResNet18_finetune_body_type')
model_resnet_bt_reg = load_model(model_resnet_bt_reg, 'ResNet18_reg_finetune_bt')

model_baseline_brand = load_model(model_baseline_brand, 'BaseLine_FC_brand')
model_cnn_1_brand = load_model(model_cnn_1_brand, 'Base_CNN_brand')
model_cnn_2_brand = load_model(model_cnn_2_brand, 'Reg_CNN_brand')
model_resnet_brand = load_model(model_resnet_brand, 'ResNet18_finetune_brand')
model_resnet_brand_reg = load_model(model_resnet_brand_reg, 'ResNet18_reg_finetune_brand')


In [35]:
def main_kolesa_test(model, test_df, img_size=(96, 160)):
    seed_everything(CFG.seed)

    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')

    kwargs = {'num_workers': CFG.num_workers, 'pin_memory': torch.cuda.is_available()}

    test_transform = transforms.Compose([
        transforms.Resize(img_size), # т.к. у нас две версии resize, то оставим как переменная
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225)
        )
        ])
    
    test_ds = KolesaDataset(test_df, transform=test_transform)

    test_loader = torch.utils.data.DataLoader(test_ds, batch_size=CFG.test_batch_size, **kwargs)

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    accuracy, macro_f1, weighted_f1, report = test(model, device, test_loader, criterion, CFG.wandb)

    return accuracy, macro_f1, weighted_f1, report

In [ ]:
CFG.wandb = False
rows = []

### Test для задачи на тип кузова

In [ ]:
CFG.classes = all_body_types
CFG.test_batch_size = 512

body_type_models = [
    ['BaseLine_FC', model_baseline_bt],
    ['Base_CNN', model_cnn_1_bt],
    ['Reg_CNN', model_cnn_2_bt],
    ['ResNet18_finetune', model_resnet_bt]
]

for model_name, model in body_type_models:
    print(model_name)
    accuracy, macro_f1, weighted_f1, report = main_kolesa_test(model, test_df_bt)

    rows.append({
        'model': model_name,
        'task': 'body_type',
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1
    })

# отдельно reg resnet

CFG.test_batch_size = 64

print('ResNet18_reg_finetune')
accuracy, macro_f1, weighted_f1, report = main_kolesa_test(model_resnet_bt_reg, test_df_bt, img_size=(320, 512))

rows.append({
    'model': 'ResNet18_reg_finetune',
    'task': 'body_type',
    'accuracy': accuracy,
    'macro_f1': macro_f1,
    'weighted_f1': weighted_f1
})

BaseLine_FC
Test set: Accuracy: 54.74%

Classification report:
              precision    recall  f1-score   support

       sedan       0.67      0.56      0.61       419
         SUV       0.41      0.52      0.46       246

    accuracy                           0.55       665
   macro avg       0.54      0.54      0.54       665
weighted avg       0.57      0.55      0.55       665

Base_CNN
Test set: Accuracy: 64.21%

Classification report:
              precision    recall  f1-score   support

       sedan       0.79      0.59      0.67       419
         SUV       0.51      0.73      0.60       246

    accuracy                           0.64       665
   macro avg       0.65      0.66      0.64       665
weighted avg       0.69      0.64      0.65       665

Reg_CNN
Test set: Accuracy: 67.52%

Classification report:
              precision    recall  f1-score   support

       sedan       0.87      0.57      0.69       419
         SUV       0.54      0.86      0.66       246



### Test для задачи на марки

In [ ]:
CFG.classes = all_brands
CFG.test_batch_size = 512

brand_models = [
    ['BaseLine_FC', model_baseline_brand],
    ['Base_CNN', model_cnn_1_brand],
    ['Reg_CNN', model_cnn_2_brand],
    ['ResNet18_finetune', model_resnet_brand]
]

for model_name, model in brand_models:
    print(model_name)
    accuracy, macro_f1, weighted_f1, report = main_kolesa_test(model, test_df_brand)

    rows.append({
        'model': model_name,
        'task': 'brand',
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1
    })

# отдельно reg resnet

CFG.test_batch_size = 64

print('ResNet18_reg_finetune')
accuracy, macro_f1, weighted_f1, report = main_kolesa_test(model_resnet_brand_reg, test_df_brand, img_size=(320, 512))

rows.append({
    'model': 'ResNet18_reg_finetune',
    'task': 'brand',
    'accuracy': accuracy,
    'macro_f1': macro_f1,
    'weighted_f1': weighted_f1
})

BaseLine_FC
Test set: Accuracy: 15.19%

Classification report:
               precision    recall  f1-score   support

         Audi       0.16      0.19      0.18        72
          BMW       0.20      0.10      0.13        90
    Chevrolet       0.16      0.15      0.16        78
      Hyundai       0.17      0.17      0.17        86
          Kia       0.12      0.10      0.11        77
        Lexus       0.15      0.15      0.15        95
Mercedes-Benz       0.15      0.13      0.14        82
       Toyota       0.13      0.21      0.16        85

     accuracy                           0.15       665
    macro avg       0.16      0.15      0.15       665
 weighted avg       0.16      0.15      0.15       665

Base_CNN
Test set: Accuracy: 19.85%

Classification report:
               precision    recall  f1-score   support

         Audi       0.24      0.32      0.28        72
          BMW       0.16      0.31      0.21        90
    Chevrolet       0.29      0.13      0.18    

In [39]:
results_dl_images = pd.DataFrame(rows)
results_dl_images

,model,task,accuracy,macro_f1,weighted_f1
0,BaseLine_FC,body_type,0.547368,0.535107,0.554748
1,Base_CNN,body_type,0.642105,0.638435,0.647912
2,Reg_CNN,body_type,0.675188,0.674652,0.678088
3,ResNet18_finetune,body_type,0.893233,0.886673,0.893766
4,ResNet18_reg_finetune,body_type,0.968421,0.965803,0.968265
5,BaseLine_FC,brand,0.151880,0.150438,0.150214
6,Base_CNN,brand,0.198496,0.189036,0.187316
7,Reg_CNN,brand,0.284211,0.260676,0.259223
8,ResNet18_finetune,brand,0.581955,0.576440,0.577551
9,ResNet18_reg_finetune,brand,0.884211,0.884551,0.883427


In [41]:
results_dl_images.to_csv('../data/results_dl_images.csv', index=False)